In [1]:
!pip install scikit-learn xgboost

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [3]:
BASE = "/kaggle/input/notebooks/krishnaiiith/02-feature-extraction"
BASE2 = "/kaggle/input/notebooks/krishnaiiith/01-dataset-preparation"

# Embeddings
text_emb = np.load(BASE + "/text_emb.npy")
img_emb  = np.load(BASE + "/img_emb.npy")
hash_emb = np.load(BASE + "/hash_emb.npy")

# Hashtag statistics
hashtag_count = np.load(BASE + "/hashtag_count.npy")
hashtag_freq  = np.load(BASE + "/hashtag_freq.npy")

# NEW (from Notebook 2)
semantic_features = np.load(BASE + "/semantic_features.npy")

# Labels
df = pd.read_csv(BASE2 + "/labeled.csv")
y = df["label"].values

In [4]:
X = np.concatenate([
    text_emb,
    img_emb,
    hash_emb,
    hashtag_count,
    hashtag_freq,
    semantic_features
], axis=1)

print("Final feature matrix shape:", X.shape)

Final feature matrix shape: (11902, 1541)


In [5]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

In [6]:
X_train_idx, X_test_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train = X[X_train_idx]
X_test  = X[X_test_idx]

y_train = y[X_train_idx]
y_test  = y[X_test_idx]

In [7]:
X = text_emb

X_train = X[X_train_idx]
X_test = X[X_test_idx]

y_train = y[X_train_idx]
y_test = y[X_test_idx]

clf = LogisticRegression(max_iter=2000, class_weight="balanced")

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

print("TEXT MODEL")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

TEXT MODEL
              precision    recall  f1-score   support

           0       0.96      0.81      0.88      2083
           1       0.36      0.75      0.49       298

    accuracy                           0.80      2381
   macro avg       0.66      0.78      0.68      2381
weighted avg       0.88      0.80      0.83      2381

ROC-AUC: 0.843580341982234


In [8]:
X = img_emb

X_train = X[X_train_idx]
X_test = X[X_test_idx]

clf = LogisticRegression(max_iter=2000, class_weight="balanced")

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

print("IMAGE MODEL")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

IMAGE MODEL
              precision    recall  f1-score   support

           0       0.93      0.75      0.83      2083
           1       0.26      0.63      0.37       298

    accuracy                           0.73      2381
   macro avg       0.60      0.69      0.60      2381
weighted avg       0.85      0.73      0.77      2381

ROC-AUC: 0.7512001920307249


In [9]:
X = hash_emb

X_train = X[X_train_idx]
X_test = X[X_test_idx]

y_train = y[X_train_idx]
y_test = y[X_test_idx]

clf = LogisticRegression(max_iter=2000, class_weight="balanced")

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

print("HASHTAG MODEL")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

HASHTAG MODEL
              precision    recall  f1-score   support

           0       0.94      0.68      0.79      2083
           1       0.24      0.71      0.36       298

    accuracy                           0.69      2381
   macro avg       0.59      0.70      0.58      2381
weighted avg       0.86      0.69      0.74      2381

ROC-AUC: 0.7414303067014213


In [10]:
X = np.concatenate([
    text_emb,
    img_emb,
    semantic_features
], axis=1)

X_train = X[X_train_idx]
X_test = X[X_test_idx]

neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)

clf = XGBClassifier(
    eval_metric="logloss",
    tree_method="hist",
    scale_pos_weight=neg/pos,
    random_state=42
)

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

print("TEXT + IMAGE + SEMANTIC MODEL")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

TEXT + IMAGE + SEMANTIC MODEL
              precision    recall  f1-score   support

           0       0.91      0.96      0.94      2083
           1       0.57      0.34      0.42       298

    accuracy                           0.89      2381
   macro avg       0.74      0.65      0.68      2381
weighted avg       0.87      0.89      0.87      2381

ROC-AUC: 0.8124679814542138


In [11]:
count = hashtag_count.reshape(-1,1)
freq = hashtag_freq.reshape(-1,1)

X = np.concatenate([
    text_emb,
    img_emb,
    hash_emb,
    semantic_features,
    count,
    freq
], axis=1)

X_train = X[X_train_idx]
X_test = X[X_test_idx]

clf = XGBClassifier(
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

print("FULL MULTIMODAL MODEL")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

FULL MULTIMODAL MODEL
              precision    recall  f1-score   support

           0       0.91      0.98      0.94      2083
           1       0.72      0.29      0.41       298

    accuracy                           0.90      2381
   macro avg       0.81      0.64      0.68      2381
weighted avg       0.88      0.90      0.88      2381

ROC-AUC: 0.8250047524382426
